## Funding analysis (2020-2025)

Computes the standard breakdowns needed for reporting, exported as CSVs sized to paste directly
into the existing Google Sheets template (`raw_data/Funding analysis 2026.xlsx`, 'Europe wide'
tab) - keeps chart formatting consistent with prior reports while the underlying numbers are
computed in code.

Methodology confirmed against that template rather than assumed:
- Funding figure throughout = `'Gov & NP contribution (EUR)'`.
- Excludes any grant whose `'Award purpose'` is *exactly* `'Equipment and infrastructure'` (not
  combined with anything else).
- "Country" = `'Funder Country'` (whose money it is), not `'PI organisation country'`.
- Multi-value cells (countries, pillars, end products) split the grant's funding evenly across
  the **distinct** values listed.
- Growth rate = simple year-over-year % change.
- Population/GDP from the World Bank, R&D spend from OECD - fetched live (see
  `Funding_analysis_helpers.py`).

In [ ]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from Funding_analysis_helpers import (
    split_multi_value, fetch_worldbank_indicator, fetch_oecd_gerd, fetch_usd_to_eur_rate,
    lookup_country_value, COUNTRY_NAME_TO_OECD, yoy_growth, YEAR_COLS,
)

FINAL_FILE = Path('data_output/260727_0922_ap_pillar_backfill_funding_curated_FINAL.xlsx')
OUTPUT_DIR = Path('data_output/analysis')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FUNDING_COL = 'Gov & NP contribution (EUR)'
YEAR_COL_STRS = [str(y) for y in YEAR_COLS]
ALL_TIME_YEARS = [str(y) for y in range(2020, 2026)]  # 2020-2025, matching the template's "All time" window

# Captured once so every export from this run shares one timestamp.
RUN_TIMESTAMP = datetime.today().strftime('%y%m%d_%H%M')


def save(df_or_series, name, **to_csv_kwargs):
    path = OUTPUT_DIR / f'{RUN_TIMESTAMP}_{name}.csv'
    df_or_series.to_csv(path, **to_csv_kwargs)
    print(f"Saved {name} -> {path}")

In [ ]:
df = pd.read_excel(FINAL_FILE)
print(f"Loaded {len(df)} rows from '{FINAL_FILE.name}'.")

excluded_mask = df['Award purpose'].astype(str).str.strip() == 'Equipment and infrastructure'
df = df[~excluded_mask].reset_index(drop=True)
print(f"Excluded {excluded_mask.sum()} grants whose Award purpose is ONLY 'Equipment and infrastructure'.")
print(f"{len(df)} grants remain for analysis.")

df['_all_time_funding'] = df[ALL_TIME_YEARS].sum(axis=1)

### 1. Total funding by start year, and by funding year

In [ ]:
by_start_year = df[df['Year project started'].between(2020, 2025)].groupby('Year project started')[FUNDING_COL].sum()
by_start_year.index = by_start_year.index.astype(int)
by_start_year.name = 'Total funding (EUR)'
save(by_start_year, '1a_funding_by_start_year')
print(by_start_year)

by_funding_year = df[YEAR_COL_STRS].sum()
by_funding_year.name = 'Total funding (EUR)'
save(by_funding_year, '1b_funding_by_funding_year')
print()
print(by_funding_year)

### 2. Total funding by year x AP pillar, plus growth rates

In [ ]:
by_year_pillar = df.groupby('AP pillar')[YEAR_COL_STRS].sum()
by_year_pillar.loc['Total'] = by_year_pillar.sum()
save(by_year_pillar, '2a_funding_by_year_and_pillar')
print(by_year_pillar)

# YoY growth rate, 2020-2025 window (matches the template's own formula: (year_N - year_N-1) / year_N-1)
growth_base = by_year_pillar[ALL_TIME_YEARS]
growth_rates = growth_base.apply(yoy_growth, axis=1)
growth_rates['Average growth rate (2021-2025)'] = growth_rates[ALL_TIME_YEARS[1:]].mean(axis=1)
save(growth_rates, '2b_growth_rates_by_pillar')
print()
print(growth_rates)

### 3. Total funding by country (2020-2025), with per-capita/GDP/R&D-spend

Fetches World Bank population/GDP and OECD R&D spend live, plus a live USD->EUR rate (Frankfurter)
to convert them - reused by analyses 4 and 8 below, so this cell's variables stay in scope.

In [ ]:
country_split = split_multi_value(df, 'Funder Country', delimiter=',')
country_split['weighted_funding'] = country_split['_all_time_funding'] * country_split['_weight']

by_country = country_split.groupby('Funder Country_split')['weighted_funding'].sum().sort_values(ascending=False)
by_country.name = 'Total funding (2020-2025, EUR)'

print("Fetching World Bank population/GDP, OECD R&D spend, and a USD->EUR rate...")
POP_GDP_YEAR = 2023  # most recent year with broad World Bank/OECD coverage
pop = fetch_worldbank_indicator('SP.POP.TOTL', POP_GDP_YEAR)
gdp_usd = fetch_worldbank_indicator('NY.GDP.MKTP.CD', POP_GDP_YEAR)
gerd_usd = fetch_oecd_gerd(POP_GDP_YEAR)
usd_to_eur = fetch_usd_to_eur_rate(f'{POP_GDP_YEAR}-12-29')
print(f"USD->EUR rate ({POP_GDP_YEAR}-12-29): {usd_to_eur}")

by_country_df = by_country.reset_index().rename(columns={'Funder Country_split': 'Country'})
by_country_df['Population'] = by_country_df['Country'].apply(lambda c: lookup_country_value(c, pop))
by_country_df['GDP (EUR)'] = by_country_df['Country'].apply(lambda c: lookup_country_value(c, gdp_usd)) * usd_to_eur
by_country_df['R&D spend (EUR)'] = by_country_df['Country'].apply(
    lambda c: lookup_country_value(c, gerd_usd, COUNTRY_NAME_TO_OECD)
) * usd_to_eur
by_country_df['Funding per capita'] = by_country_df['Total funding (2020-2025, EUR)'] / by_country_df['Population']
by_country_df['Funding per GDP'] = by_country_df['Total funding (2020-2025, EUR)'] / by_country_df['GDP (EUR)']
by_country_df['GDP per capita'] = by_country_df['GDP (EUR)'] / by_country_df['Population']
by_country_df['Funding per R&D spend'] = by_country_df['Total funding (2020-2025, EUR)'] / by_country_df['R&D spend (EUR)']

n_no_population = by_country_df['Population'].isna().sum()
print(f"{n_no_population} entries have no population match (expected for non-country entities like 'EU'/'Global').")

save(by_country_df, '3_funding_by_country', index=False)
print(by_country_df.head(15).to_string())

### 4. Total funding by country x AP pillar (2020-2025)

In [ ]:
country_pillar = country_split.pivot_table(
    index='Funder Country_split', columns='AP pillar', values='weighted_funding', aggfunc='sum', fill_value=0
)
country_pillar['Total'] = country_pillar.sum(axis=1)
country_pillar = country_pillar.sort_values('Total', ascending=False)
save(country_pillar, '4_funding_by_country_and_pillar')
print(country_pillar)

### 5. Top 20 funder institutions, with yearly split (no pillar split)

In [ ]:
by_funder = df.groupby('Funder name')[YEAR_COL_STRS].sum()
by_funder['Total'] = by_funder.sum(axis=1)
top20_funders = by_funder.sort_values('Total', ascending=False).head(20)
save(top20_funders, '5_top20_funders')
print(top20_funders)

### 6. Top 20 country rankings per year, 2020-2025 (non-cumulative)

Two shapes: (a) rank 1-20 x year, country name per cell (for a table); (b) country x year, rank
number per cell (for graphing a ranked-line/bump chart). Distinct from analysis #8's cumulative
ranking below - confirmed via the template these are two different things, not variants of one.

In [ ]:
annual_country = country_split.copy()
year_country_totals = {}
for y in range(2020, 2026):
    ycol = str(y)
    annual_country['_year_amount'] = annual_country[ycol] * annual_country['_weight']
    year_country_totals[y] = annual_country.groupby('Funder Country_split')['_year_amount'].sum().sort_values(ascending=False)

# Shape A: rank (1-20) x year, country name per cell
rank_table = pd.DataFrame({y: year_country_totals[y].head(20).index.tolist() for y in range(2020, 2026)})
rank_table.index = pd.RangeIndex(1, 21, name='Rank')
save(rank_table, '6a_country_rank_table')
print(rank_table)

# Shape B: country x year, rank number per cell (for graphing)
all_countries_ranked = sorted(set().union(*[year_country_totals[y].head(20).index for y in range(2020, 2026)]))
country_rank_grid = pd.DataFrame(index=all_countries_ranked, columns=range(2020, 2026), dtype='float')
for y in range(2020, 2026):
    ranks = {c: i + 1 for i, c in enumerate(year_country_totals[y].index)}
    for c in all_countries_ranked:
        country_rank_grid.loc[c, y] = ranks.get(c, np.nan)
save(country_rank_grid, '6b_country_rank_grid')
print()
print(country_rank_grid)

### 7. Total funding 2020-2025 by AP pillar x End product type, plus a separate Dairy sub-split

In [ ]:
ep_split = split_multi_value(df, 'End product type', delimiter=';')
ep_split['weighted_funding'] = ep_split['_all_time_funding'] * ep_split['_weight']
pillar_end_product = ep_split.pivot_table(
    index='End product type_split', columns='AP pillar', values='weighted_funding', aggfunc='sum', fill_value=0
)
pillar_end_product['Total'] = pillar_end_product.sum(axis=1)
save(pillar_end_product, '7a_funding_by_pillar_and_end_product')
print(pillar_end_product)

# Dairy's own constituents, not cross-tabulated with pillar
sub_ep_split = split_multi_value(df, 'sub-end product', delimiter=';')
sub_ep_split['weighted_funding'] = sub_ep_split['_all_time_funding'] * sub_ep_split['_weight']
dairy_breakdown = sub_ep_split.groupby('sub-end product_split')['weighted_funding'].sum().sort_values(ascending=False)
dairy_breakdown.name = 'Total funding (2020-2025, EUR)'
save(dairy_breakdown, '7b_dairy_sub_breakdown')
print()
print(dairy_breakdown)

### 8. Investment and cumulative investment by country, 2020-2025, with cumulative rank

Reuses `year_country_totals` computed for analysis #6 above.

In [ ]:
investment_by_year = pd.DataFrame({y: year_country_totals[y] for y in range(2020, 2026)}).fillna(0)
save(investment_by_year, '8a_investment_by_country_and_year')

cumulative_by_year = investment_by_year.cumsum(axis=1)
save(cumulative_by_year, '8b_cumulative_investment_by_country_and_year')

# Rank 1 = highest cumulative total-to-date that year
cumulative_rank = cumulative_by_year.rank(axis=0, ascending=False, method='min')
save(cumulative_rank, '8c_cumulative_investment_rank')

print(investment_by_year.head(10))
print()
print(cumulative_by_year.head(10))
print()
print(cumulative_rank.head(10))